# Step C on Kaggle: continue the 15 training runs where Colab stopped

Same code and protocol as `training/train_colab.ipynb` (DEC-005/006/007). It restores every saved run from the inputs, so finished runs are skipped and an interrupted one resumes from its last epoch.

**One-time setup**
1. Kaggle account with a **verified phone number** (needed for GPU and Internet).
2. In Google Drive, right-click `Argus/train_v3` → **Download** (a zip of ~350 MB).
3. Kaggle → **Datasets → New Dataset** → upload that zip → title `argus-train-v3` → Create.
4. In this notebook: **+ Add Input** → add the dataset `awsaf49/brats20-dataset-training-validation` and your `argus-train-v3`.
5. **Settings** (right panel): Accelerator **GPU T4 x2**, **Internet on**.

**Run it in the background:** **Save Version → Save & Run All (Commit)**. It keeps running with the browser closed (Kaggle limit: 12 h per run). The loop stops starting new runs after 10.5 h so the outputs are always saved. If not all 15 are done, open the notebook again, **+ Add Input → this notebook's output**, and commit again: it continues from there.

In [ ]:
import time; T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
# ---- Configuration ----
import glob, os
RAW    = glob.glob('/kaggle/input/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]   # BraTS, already extracted
WORK   = '/tmp/cache'                       # prepared arrays (scratch disk, not saved)
OUT    = '/kaggle/working/train_v3'         # checkpoints + logs (saved as notebook output)
MODELS = ['M0', 'M1', 'M4', 'M5', 'M6']
SEEDS  = [0, 1, 2]
EPOCHS = 20
TIME_LIMIT_H = 10.5                         # do not START a new run after this (Kaggle stops at 12 h)
REPO, BRANCH = 'https://github.com/joshykoshy/Argus.git', 'matlab-analysis'
os.makedirs(OUT, exist_ok=True); print('BraTS at', RAW)

In [ ]:
!git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/temp_repo 2>/dev/null || git -C /kaggle/temp_repo pull -q
%cd /kaggle/temp_repo
!pip -q install nibabel

In [ ]:
# Restore saved runs from ALL inputs (Drive download dataset and/or a previous version's output).
# For each model/seed keep the copy with the most epochs done, so nothing already trained is repeated.
import shutil, pandas as pd
best = {}
for log in glob.glob('/kaggle/input/**/seed_*/log.csv', recursive=True):
    run = os.path.dirname(log)
    model, seed = run.split('/')[-2], run.split('/')[-1]
    if model not in MODELS or not os.path.exists(f'{run}/last.pt'):
        continue
    ep = int(pd.read_csv(log).epoch.max())
    if ep > best.get((model, seed), (-1, None))[0]:
        best[(model, seed)] = (ep, run)
for (model, seed), (ep, run) in sorted(best.items()):
    dst = f'{OUT}/{model}/{seed}'
    have = int(pd.read_csv(f'{dst}/log.csv').epoch.max()) if os.path.exists(f'{dst}/log.csv') else -1
    if ep > have:
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(run, dst)
    print(f'{model} {seed}: {max(ep, have)}/{EPOCHS} epochs restored')
if not best: print('No saved runs found in the inputs - check that argus-train-v3 is added as input')

In [ ]:
# Preprocess into memory-mappable arrays (~20-30 min; reads BraTS straight from the Kaggle input)
!python -m training.prepare --raw_dir {RAW} --work {WORK}
!du -sh {WORK}

In [ ]:
# Train: seed by seed; finished runs are skipped, an interrupted one resumes.
# No new run starts after TIME_LIMIT_H, so the version always finishes and saves its output.
for seed in SEEDS:
    for model in MODELS:
        log = f'{OUT}/{model}/seed_{seed}/log.csv'
        if os.path.exists(log) and pd.read_csv(log).epoch.max() >= EPOCHS:
            continue
        hours = (time.time() - T0) / 3600
        if hours > TIME_LIMIT_H:
            print(f'Stopping at {hours:.1f} h: commit again with this output as input to continue.'); break
        !python -m training.train_v2 --model {model} --seed {seed} --epochs {EPOCHS} --work {WORK} --out {OUT}
    else:
        continue
    break

In [ ]:
# Results table (best validation epoch of each run)
rows = []
for f in sorted(glob.glob(f'{OUT}/*/seed_*/log.csv')):
    full = pd.read_csv(f); log = full.dropna(subset=['val_mean'])
    if len(log):
        b = log.loc[log.val_mean.idxmax()]
        rows.append(dict(model=f.split('/')[-3], seed=f.split('/')[-2], epochs_done=int(full.epoch.max()),
                         best_epoch=int(b.epoch), clean=b['val_clean'], snr5=b['val_kspace_snr5_r1.0_t1'],
                         worst=b['val_kspace_snr3_r0.5_t5'], mean=b.val_mean, false_tumor_cm3=b.val_false_tumor_cm3))
R = pd.DataFrame(rows); print(R.to_string(index=False)); R.to_csv(f'{OUT}/stepC_summary.csv', index=False)

**When it finishes:** open the version → **Output** tab → download `train_v3` and upload it to Google Drive `Argus/train_v3` (replace), or just tell me the results table. Then comes Step D: evaluation on the 74 test patients.